### Membuat SparkSession

In [15]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, count, avg, when

spark = (
    SparkSession.builder
    .appName("Tugas4-Gilang-EcommerceSeptember")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession aktif, versi Spark:", spark.version)

SparkSession aktif, versi Spark: 3.5.9


### Menyiapkan Dataset

In [16]:
import numpy as np
import pandas as pd

np.random.seed(99)
total_baris = 1000

daftar_kategori = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga", "Olahraga"]
daftar_kota = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo", "Kebumen"]
daftar_metode_bayar = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
rentang_tanggal = pd.date_range("2026-09-01", "2026-09-30", freq="D")

data_mentah = {
    "order_id": [f"ORD-{3000 + i}" for i in range(total_baris)],
    "tanggal": np.random.choice(rentang_tanggal, size=total_baris).astype(str),
    "kategori": np.random.choice(daftar_kategori, size=total_baris),
    "kota": np.random.choice(daftar_kota, size=total_baris),
    "unit_terjual": np.random.randint(1, 12, size=total_baris),
    "harga_satuan": np.random.choice([20000, 45000, 60000, 90000, 125000, 200000, 350000], size=total_baris),
    "metode_pembayaran": np.random.choice(daftar_metode_bayar, size=total_baris),
    "rating": np.random.choice([1, 2, 3, 4, 5, np.nan], size=total_baris, p=[0.03, 0.02, 0.10, 0.30, 0.35, 0.20]),
}
df_pandas = pd.DataFrame(data_mentah)
df_pandas.to_csv("transaksi_september_2026.csv", index=False)
print(f"Dataset lokal dibuat: {df_pandas.shape[0]} baris, {df_pandas.shape[1]} kolom")

# Diunggah ke folder HDFS pribadi (username: xiuviu)
!hdfs dfs -mkdir -p /user/xiuviu/tugas4
!hdfs dfs -put -f transaksi_september_2026.csv /user/xiuviu/tugas4/
print("Dataset berhasil diunggah ke: /user/xiuviu/tugas4/transaksi_september_2026.csv")

Dataset lokal dibuat: 1000 baris, 8 kolom
Dataset berhasil diunggah ke: /user/xiuviu/tugas4/transaksi_september_2026.csv


### Membaca dan Eksplorasi Awal

In [17]:
path_hdfs = "hdfs://localhost:9000/user/xiuviu/tugas4/transaksi_september_2026.csv"

df = spark.read.csv(path_hdfs, header=True, inferSchema=True)

print("--- Struktur kolom (schema) ---")
df.printSchema()

print("Total baris pada dataset:", df.count())

print("--- 10 baris pertama ---")
df.show(10)

--- Struktur kolom (schema) ---
root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)

Total baris pada dataset: 1000
--- 10 baris pertama ---
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|

### Menangani Data Kosong

In [18]:
baris_rating_kosong = df.filter(col("rating").isNull()).count()
total_baris_awal = df.count()
persentase_kosong = round(baris_rating_kosong / total_baris_awal * 100, 2)

print(f"Baris dengan rating kosong : {baris_rating_kosong} dari {total_baris_awal} ({persentase_kosong}%)")

Baris dengan rating kosong : 204 dari 1000 (20.4%)


**Pilihan saya: `df.na.drop()`, bukan `df.na.fill()`.**

Alasan: proporsi baris dengan `rating` kosong relatif kecil (di bawah 20% dari data), dan `rating` termasuk kolom yang akan langsung dipakai untuk analisis rata-rata di bagian D (bukan sekadar kolom pelengkap). Jika nilai kosong diisi dengan angka buatan (misalnya rata-rata), hasil rata-rata rating per metode pembayaran di bagian D3 justru akan "ditarik" mendekati nilai isian tersebut dan menutupi pola asli dari pelanggan yang benar-benar memberi rating. Karena jumlah baris yang hilang masih wajar dan kolom-kolom lain (`total_pendapatan`, `kategori`, `kota`) pada baris yang dibuang tidak dipakai untuk agregasi per rating, saya pilih membuang baris tersebut hanya berdasarkan kolom `rating` saja (bukan drop total) agar baris yang masih relevan untuk analisis lain tidak ikut hilang.

In [19]:
df = df.na.drop(subset=["rating"])

sisa_baris = df.count()
print("Jumlah baris setelah drop rating kosong:", sisa_baris)
print("Verifikasi rating kosong tersisa:", df.filter(col("rating").isNull()).count())

Jumlah baris setelah drop rating kosong: 796
Verifikasi rating kosong tersisa: 0


### Transformasi Data

In [20]:
df = df.withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan"))

df = df.withColumn(
    "tier_transaksi",
    when(col("total_pendapatan") > 500000, "Besar").otherwise("Kecil")
)

df.select("order_id", "kategori", "unit_terjual", "harga_satuan", "total_pendapatan", "tier_transaksi").show(10)

+--------+--------------------+------------+------------+----------------+--------------+
|order_id|            kategori|unit_terjual|harga_satuan|total_pendapatan|tier_transaksi|
+--------+--------------------+------------+------------+----------------+--------------+
|ORD-3000|        Rumah Tangga|           3|       90000|          270000|         Kecil|
|ORD-3001|   Makanan & Minuman|           3|      200000|          600000|         Besar|
|ORD-3002|Kesehatan & Kecan...|           8|       60000|          480000|         Kecil|
|ORD-3003|   Makanan & Minuman|           6|      350000|         2100000|         Besar|
|ORD-3004|        Rumah Tangga|          10|       60000|          600000|         Besar|
|ORD-3005|             Fashion|           5|       20000|          100000|         Kecil|
|ORD-3006|   Makanan & Minuman|           2|       20000|           40000|         Kecil|
|ORD-3008|             Fashion|           7|       20000|          140000|         Kecil|
|ORD-3009|

### D.1 Analisis dengan GroupBy

In [21]:
rekap_kategori = (
    df.groupBy("kategori")
      .agg(spark_sum("total_pendapatan").alias("total_pendapatan"))
      .orderBy(col("total_pendapatan").desc())
)

rekap_kategori.show()

kategori_teratas = rekap_kategori.first()["kategori"]
print("Jawaban: kategori dengan total_pendapatan tertinggi adalah", kategori_teratas)

+--------------------+----------------+
|            kategori|total_pendapatan|
+--------------------+----------------+
|        Rumah Tangga|       108285000|
|   Makanan & Minuman|       106085000|
|             Fashion|       101725000|
|Kesehatan & Kecan...|        98040000|
|            Olahraga|        94230000|
|          Elektronik|        89325000|
+--------------------+----------------+

Jawaban: kategori dengan total_pendapatan tertinggi adalah Rumah Tangga


### D.2 Kota dengan jumlah transaksi tier Besar terbanyak

In [22]:
rekap_tier_besar = (
    df.filter(col("tier_transaksi") == "Besar")
      .groupBy("kota")
      .agg(count("order_id").alias("jumlah_tier_besar"))
      .orderBy(col("jumlah_tier_besar").desc())
)

rekap_tier_besar.show()

kota_teratas = rekap_tier_besar.first()["kota"]
print("Jawaban: kota dengan transaksi tier Besar terbanyak adalah", kota_teratas)

+----------+-----------------+
|      kota|jumlah_tier_besar|
+----------+-----------------+
|      Solo|               74|
|   Kebumen|               63|
|Yogyakarta|               61|
| Purworejo|               55|
|  Magelang|               52|
|  Semarang|               47|
+----------+-----------------+

Jawaban: kota dengan transaksi tier Besar terbanyak adalah Solo


### D.3 Rata-rata rating per metode_pembayaran

In [23]:
rekap_rating_metode = (
    df.groupBy("metode_pembayaran")
      .agg(avg("rating").alias("rata_rata_rating"))
      .orderBy(col("rata_rata_rating").desc())
)

rekap_rating_metode.show()

+-----------------+-----------------+
|metode_pembayaran| rata_rata_rating|
+-----------------+-----------------+
|              COD|4.172413793103448|
|    Transfer Bank| 4.16256157635468|
|         E-Wallet|4.135678391959799|
|     Kartu Kredit|4.109947643979058|
+-----------------+-----------------+



Karena baris dengan `rating` kosong sudah dibuang sejak bagian B, rata-rata di atas dihitung murni dari transaksi yang memang diberi rating oleh pembeli, tanpa tercampur nilai buatan hasil imputasi.

### Menyimpan Hasil ke HDFS 

In [24]:
path_output = "hdfs://localhost:9000/user/xiuviu/tugas4/hasil_olahan"

df.write.mode("overwrite").option("header", True).csv(path_output)
print("DataFrame berhasil disimpan ke:", path_output)

DataFrame berhasil disimpan ke: hdfs://localhost:9000/user/xiuviu/tugas4/hasil_olahan


In [25]:
!hdfs dfs -ls /user/xiuviu/tugas4/hasil_olahan

Found 2 items
-rw-r--r--   3 xiuviu supergroup          0 2026-09-17 00:44 /user/xiuviu/tugas4/hasil_olahan/_SUCCESS
-rw-r--r--   3 xiuviu supergroup      77337 2026-09-17 00:44 /user/xiuviu/tugas4/hasil_olahan/part-00000-f15c657a-0d02-461a-8b65-8f58391a1b99-c000.csv


**Kenapa hasilnya berupa banyak berkas `part-00000...` dan seterusnya, bukan satu file CSV utuh seperti `pandas.to_csv()`?**

Karena Spark bekerja secara terdistribusi: DataFrame yang saya olah dibagi menjadi beberapa partisi, dan setiap partisi ditulis oleh task/executor-nya masing-masing secara paralel, tanpa harus menunggu partisi lain selesai atau dikumpulkan dulu ke satu proses tunggal. Itu sebabnya keluarannya berupa banyak berkas `part-xxxxx` sekaligus — jumlahnya mengikuti jumlah partisi DataFrame saat proses tulis dijalankan. Ini justru wajar dan mencerminkan cara Spark menskalakan proses I/O untuk data yang jauh lebih besar daripada yang bisa ditangani satu mesin/proses saja.

### Menutup SparkSession

In [26]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
